# TradeFlow AI — nb3_olm_finetune (v2 — 4-Phase Curriculum)

**Model**: `allenai/olmOCR-7B-0225-preview` + LoRA fine-tuning  
**GPU**: Kaggle T4×2 (30h/minggu gratis — cukup untuk 3–4 run)

---

## Strategi Anti-Memorisasi (Ringkasan)

| Phase | Data | LR | LoRA r | Tujuan |
|---|---|---|---|---|
| **0: DAPT** | 8 gambar asli (unsupervised) | 1e-4 | - | Visual encoder adaptasi ke pixel asli |
| **1: Synthetic SFT** | 1.500 CIPL sintetis | 2e-4 | 32 | Belajar struktur field |
| **2: Mixed** | 80% sintetis + 20% real aug | 1e-4 | 32 | Menutup distribution gap |
| **3: Hard Neg** | Kasus sulit dari real docs | 5e-5 | 16 | Denoising & fine-grained |

**Early stopping**: Berbasis ANLS pada 3 dokumen val asli — bukan eval_loss sintetis.

In [ ]:
!pip install -q peft transformers datasets accelerate bitsandbytes
!pip install -q pdf2image pillow albumentations
!apt-get install -qq poppler-utils

In [ ]:
import json, os, math, time
import torch
import numpy as np
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset, load_dataset

MODEL_ID  = 'allenai/olmOCR-7B-0225-preview'
OUT_DIR   = Path('./olmocr-tradeflow-lora')
DATA_DIR  = Path('./dataset')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Konfigurasi Field-Weighted Loss

Field CEISA-kritis mendapat penalti lebih besar jika salah diprediksi.

In [ ]:
# Bobot per field (lihat TRAINING_STRATEGY.md untuk rationale)
FIELD_WEIGHTS = {
    'nomorBl':      3.0,  # kunci primer CEISA
    'hs_code':      3.0,  # penyebab penolakan #1
    'beratKotor':   2.5,  # penyebab penolakan #2
    'tglBl':        2.0,  # error format tanggal sering terjadi
    'container_no': 2.0,  # error normalisasi spasi
    'pelabuhan_muat': 1.5,
    'pelabuhan_bongkar': 1.5,
}
DEFAULT_WEIGHT = 1.0

print('Field weights configured:')
for f, w in FIELD_WEIGHTS.items():
    print(f'  {f:22} → {w}')

In [ ]:
# ── Metrics helper: ANLS (Average Normalized Levenshtein Similarity) ─────────
def anls(pred: str, gold: str, threshold: float = 0.5) -> float:
    """Standard ANLS metric (VQA evaluation standard)."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    pred, gold = str(pred).lower().strip(), str(gold).lower().strip()
    # Normalised edit distance
    m, n = len(pred), len(gold)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + (0 if pred[i-1] == gold[j-1] else 1)
            )
    dist = dp[m][n]
    sim  = 1.0 - dist / max(m, n)
    return sim if sim >= threshold else 0.0


def eval_on_real_docs(model, tokenizer, manifest_path: Path, gt_path: Path) -> float:
    """
    Hitung rata-rata ANLS pada validation docs asli.
    Ini digunakan sebagai early-stopping signal — bukan eval_loss sintetis.

    Returns: mean ANLS (0.0–1.0)
    """
    if not manifest_path.exists() or not gt_path.exists():
        print('  [WARNING] Manifest atau GT tidak ditemukan, skip eval')
        return 0.0

    manifest = json.loads(manifest_path.read_text())
    gt_data  = json.loads(gt_path.read_text())

    val_items = manifest.get('val', [])
    if not val_items:
        return 0.0

    scores = []
    model.eval()
    with torch.no_grad():
        for item in val_items[:10]:  # Ambil 10 sampel untuk kecepatan
            from PIL import Image
            img_path = Path(item['path'])
            if not img_path.exists():
                continue

            doc_id = item['doc_id']
            gt = gt_data.get(doc_id, {})
            if not gt:
                continue

            # Prompt untuk ekstraksi field
            prompt = (
                'Extract CEISA fields from this shipping document. '
                'Return JSON with: nomorBl, tglBl, pelabuhan_muat, '
                'pelabuhan_bongkar, container_no, beratKotor, hs_code.'
            )
            inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

            # Forward pass
            try:
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.1,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )
                pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

                # Coba parse JSON dari output
                try:
                    start = pred_text.find('{')
                    end   = pred_text.rfind('}') + 1
                    pred_json = json.loads(pred_text[start:end]) if start >= 0 else {}
                except (ValueError, json.JSONDecodeError):
                    pred_json = {}

                # Hitung ANLS per field (dengan field weighting)
                field_scores = []
                for field in FIELD_WEIGHTS:
                    pred_val = str(pred_json.get(field, ''))
                    gold_val = str(gt.get(field, ''))
                    if gold_val:
                        w = FIELD_WEIGHTS.get(field, DEFAULT_WEIGHT)
                        field_scores.extend([anls(pred_val, gold_val)] * int(w))
                if field_scores:
                    scores.append(np.mean(field_scores))
            except Exception as e:
                print(f'    [Eval error] {doc_id}: {e}')

    mean_anls = float(np.mean(scores)) if scores else 0.0
    print(f'  → Validation ANLS: {mean_anls:.4f} (n={len(scores)})')
    return mean_anls

print('Metric helpers siap')

## Load Model & Tokenizer

In [ ]:
print(f'Loading tokenizer dari {MODEL_ID}...')
# NOTE: Uncomment untuk run asli di Kaggle GPU
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     load_in_4bit=True,
#     device_map='auto',
#     torch_dtype=torch.float16
# )
# model = prepare_model_for_kbit_training(model)
print('Model loading disimulasikan (uncomment untuk GPU asli)')
tokenizer, model = None, None

## Phase 0: Domain Adaptive Pre-Training (DAPT)

**Prinsip**: Jangan kasih GT labels. Cukup biarkan model 'melihat' dokumen asli sehingga visual encoder-nya adapt.

In [ ]:
print('=== Phase 0: DAPT ===')
print('Tujuan: Adaptasi visual encoder ke distribusi dokumen asli')
print('Data  : 8 gambar asli (unsupervised — TANPA GT labels)')
print('Epoch : 3  |  LR: 1e-4  |  Tidak ada LoRA')
print()

# DAPT config (causal LM saja, tanpa label ekstraksi field)
dapt_args = {
    'learning_rate': 1e-4,
    'num_train_epochs': 3,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    # Tidak ada LoRA di phase 0
}

# NOTE: Uncomment untuk run asli
# train_dapt(model, tokenizer, augmented_manifest, dapt_args)
print('[SIMULATED] Phase 0 DAPT selesai')

## Phase 1: Synthetic SFT dengan LoRA

Training supervised pertama — 1.500 CIPL sintetis dari nb1.

In [ ]:
print('=== Phase 1: Synthetic SFT ===')
print('Data : 1.500 CIPL sintetis (output nb1)')
print('Epoch: 3  |  LR: 2e-4  |  LoRA r=32, α=64, dropout=0.10')
print()

lora_config_p1 = LoraConfig(
    r=32,            # rank lebih tinggi dari default r=16 → generalisasi lebih baik
    lora_alpha=64,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.10,  # lebih tinggi dari default 0.05 → regularisasi lebih kuat
    bias='none',
    task_type='CAUSAL_LM'
)

phase1_args = {
    'learning_rate': 2e-4,
    'num_train_epochs': 3,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'label_smoothing_factor': 0.1,  # anti-memorisasi
}

# NOTE: Uncomment untuk run asli
# model = get_peft_model(model, lora_config_p1)
# train_synthetic_sft(model, tokenizer, synthetic_dataset, phase1_args)
print('[SIMULATED] Phase 1 Synthetic SFT selesai')
print('Checkpoint disimpan: ./olmocr-tradeflow-lora/phase1/')

## Phase 2: Mixed Training (80% Synthetic + 20% Real)

**Ini adalah fase terpenting untuk anti-memorisasi.**  
Tidak ada mini-batch yang 100% sintetis — selalu ada dokumen asli yang di-augment.

In [ ]:
print('=== Phase 2: Mixed Training ===')
print('Data : 80% sintetis + 20% augmented real (tidak ada all-synthetic batch!)')
print('Epoch: 3  |  LR: 1e-4  |  LoRA r=32 (dilanjutkan dari Phase 1)')
print()

REAL_DOC_RATIO = 0.20  # setidaknya 20% dokumen asli per batch

phase2_args = {
    'learning_rate': 1e-4,      # setengah dari phase 1 — fine-tuning
    'num_train_epochs': 3,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'label_smoothing_factor': 0.1,
}

# Early stopping berbasis ANLS pada real val docs — BUKAN eval_loss sintetis!
best_anls  = 0.0
patience   = 0
MAX_PATIENCE = 2

# Simulasi loop training dengan early stopping
for epoch in range(3):
    print(f'  Epoch {epoch+1}/3 [Mixed Training]')
    # train_epoch_mixed(model, tokenizer, synthetic_ds, augmented_ds, REAL_DOC_RATIO, phase2_args)

    # Evaluasi pada real val docs
    # val_anls = eval_on_real_docs(model, tokenizer, MANIFEST_PATH, GT_PATH)
    val_anls = 0.78 + epoch * 0.03  # SIMULASI — hapus di run asli

    if val_anls > best_anls:
        best_anls = val_anls
        patience  = 0
        print(f'    ✓ ANLS baru terbaik: {best_anls:.4f} — checkpoint disimpan')
        # model.save_pretrained('./olmocr-tradeflow-lora/best/')
    else:
        patience += 1
        print(f'    ✗ ANLS tidak membaik ({val_anls:.4f}), patience={patience}/{MAX_PATIENCE}')
        if patience >= MAX_PATIENCE:
            print('  Early stop: val ANLS sudah plateau pada real docs!')
            break

print(f'\n[SIMULATED] Phase 2 selesai. Best val ANLS: {best_anls:.4f}')

## Phase 3: Hard Negatives — Kasus Sulit

Fokus pada kasus-kasus di mana synthetic dan real paling berbeda.

In [ ]:
print('=== Phase 3: Hard Negatives ===')
print('Data : Kasus sulit — tanggal ambigu, nomor kontainer berspace, berat MTS')
print('Epoch: 2  |  LR: 5e-5  |  LoRA r=16 (lebih ketat)')
print()

# Hard negative categories (dari analisis distribusi gap)
HARD_NEGATIVE_TYPES = [
    'date_format_ambiguous',    # Tanggal bisa dibaca dua cara (03/04/2024)
    'container_with_space',     # HLXU 2382861 vs HLXU2382861
    'weight_in_mts',            # MTS harus dikali 1000 → beratKotor
    'hs_code_with_dots',        # 8482.10.00 harus dinormalisasi ke 84821000
    'overlapping_watermark',    # Watermark menutupi field penting
    'rotated_table',            # Tabel miring ±5°
    'multipage_cargo',          # Kargo tersebar di 2 halaman
]

lora_config_p3 = LoraConfig(
    r=16,         # rank lebih rendah → capacity lebih kecil → overfitting lebih kecil
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.10,
    bias='none',
    task_type='CAUSAL_LM'
)

phase3_args = {
    'learning_rate': 5e-5,   # sangat kecil — fine-grained correction saja
    'num_train_epochs': 2,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'label_smoothing_factor': 0.1,
}

print(f'Hard negative types: {len(HARD_NEGATIVE_TYPES)}')
for hn in HARD_NEGATIVE_TYPES:
    print(f'  - {hn}')

# NOTE: Uncomment untuk run asli
# train_hard_negatives(model, tokenizer, hard_neg_dataset, phase3_args)
print('\n[SIMULATED] Phase 3 Hard Negatives selesai')

## Upload Model ke HuggingFace Hub

In [ ]:
# Setelah semua phase selesai, upload adapter ke HuggingFace
# Adapter inilah yang akan di-download otomatis oleh Docker container
# saat startup berdasarkan OLM_LORA_ADAPTER di .env

HF_REPO_NAME = 'your-org/olm-ocr-cipl-v1'  # Ganti dengan org HF Anda
HF_TOKEN     = os.environ.get('HF_TOKEN', 'hf_...')  # Dari Kaggle Secrets

print(f'Upload target: {HF_REPO_NAME}')
print('Pastikan HF_TOKEN ada di Kaggle Secrets!')

# NOTE: Uncomment untuk upload asli
# from huggingface_hub import HfApi
# api = HfApi(token=HF_TOKEN)
# api.create_repo(repo_id=HF_REPO_NAME, exist_ok=True)
# model.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
# print(f'✓ Model berhasil diupload ke: https://huggingface.co/{HF_REPO_NAME}')

print('\n[SIMULATED] Upload ke HuggingFace selesai')
print(f'Update .env: OLM_LORA_ADAPTER={HF_REPO_NAME}')
print('Docker container akan auto-download adapter ini saat startup!')

## Ringkasan

```
Teknik Anti-Memorisasi yang Digunakan:
✓ LoRA r=32 (Phase 1-2) → capacity terbatas, generalisasi lebih baik
✓ Dropout 0.10 → regularisasi lebih kuat
✓ Mixed training 80/20 → real docs selalu hadir
✓ Early stopping berbasis real-doc ANLS → tidak bisa hafal sintetis
✓ Label smoothing 0.1 → prediksi tidak terlalu confident pada sintetis
✓ DAPT Phase 0 → visual encoder sudah adaptasi ke real pixel sebelum SFT
✓ Hard negative Phase 3 → fokus pada kasus yang paling sulit
```

**Lanjutkan ke nb4_eval.ipynb untuk evaluasi final! →**